In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.retrievers import PineconeHybridSearchRetriever
from pinecone import Pinecone, ServerlessSpec

index_name = "hybrid-search-implementation"
pc = Pinecone()

if index_name not in pc.list_indexes().names():
  pc.create_index(
    name=index_name,
    dimension=384,
    metric="dotproduct",
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
  )

In [3]:
index = pc.Index(index_name)
index

Index(host='https://hybrid-search-implementation-ylczpxp.svc.aped-4627-b74a.pinecone.io')

In [4]:
# Vector Embedding and Sparse Matrix
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1668.49it/s]


HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [8]:
from pinecone_text.sparse import BM25Encoder

encoder =BM25Encoder().default()
encoder

In [18]:
sentences = [
  "Hello, My name is Anand Mansabdar",
  "I am currently learning hybrid search with pinecone db",
  "I want to become an AI ENgineer"
]

encoder.fit(sentences)

encoder.dump("bm25_values.json")

encoder = BM25Encoder().load("bm25_values.json")


100%|██████████| 3/3 [00:00<00:00, 1499.39it/s]


In [21]:
retriever = PineconeHybridSearchRetriever(embeddings=embeddings, sparse_encoder=encoder, index=index)

In [22]:
retriever

PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x00000257D0A3BE50>, index=Index(host='https://hybrid-search-implementation-ylczpxp.svc.aped-4627-b74a.pinecone.io'))

In [ ]:
retriever.add_texts(texts=["I am Anand Mansabdar"])
retriever.invoke("What am I currently learning?")